# Kaggriculture: PPO против Boatlee V16-RC5

Эта тетрадь скачивает публичного агента Boatlee через Kaggle, обучает нейросетевой контроллер играть против него, проверяет результат на отдельных seed, сохраняет всё на Google Drive и собирает `submission.tar.gz`.

Контроллер выбирает один из четырёх вариантов каждого хода: исходный эксперт, наш rule-based агент и два их гибрида. Поэтому даже до улучшения сохранённый агент не слабее исходного Boatlee на контрольной серии. Достижение заданного процента побед не гарантируется: запуск ограничен числом раундов, а отправка автоматически блокируется, если цель не достигнута.

Перед **Runtime → Run all** добавьте `KAGGLE_API_TOKEN` в **Colab → Secrets**. Запрос Google Drive появляется в первой ячейке, до долгого обучения.

In [ ]:
# Все параметры, которые обычно нужно менять.
STEPS_PER_ROUND = 100_000
MAX_ROUNDS = 10
TARGET_WIN_RATE = 0.80
EVAL_GAMES = 20
TRAIN_ENVS = 2
SUBMIT_TO_KAGGLE = True
SUBMIT_BELOW_TARGET = False
FORCE_SUBMIT = False
SUBMISSION_MESSAGE = "Boatlee V16 expert-gated PPO v1"

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive/Kaggriculture/results")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Google Drive подключён: {DRIVE_ROOT}")

In [ ]:
# Загружаем актуальный код и проверяем Kaggle до долгой работы.
import os
import subprocess
import sys
from google.colab import userdata

REPOSITORY = "https://github.com/GrigoriiIurev/Kaggriculture.git"
PROJECT = Path("/content/Kaggriculture")
if (PROJECT / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-rl.txt", "kaggle"], check=True)

try:
    token = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    token = None
if token:
    os.environ["KAGGLE_API_TOKEN"] = token
check = subprocess.run(["kaggle", "competitions", "list", "-s", "kaggriculture"], text=True, capture_output=True)
if check.returncode != 0:
    raise RuntimeError("Kaggle не авторизован. Добавьте KAGGLE_API_TOKEN в Colab Secrets.\n" + check.stderr)
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Kaggle авторизован, версия кода: {commit}")

## Обучение, проверка и условная отправка

Один раунд в настройках выше содержит 100 000 игровых ходов, то есть примерно 139 полных игр. После каждого раунда идут 20 контрольных игр с чередованием стороны. Модель и отчёт сохраняются на Drive; после перезапуска обучение продолжится с последней точки.

In [ ]:
command = [
    sys.executable, "-u", "run_rl_colab_pipeline.py",
    "--drive-root", str(DRIVE_ROOT),
    "--steps-per-round", str(STEPS_PER_ROUND),
    "--max-rounds", str(MAX_ROUNDS),
    "--target-win-rate", str(TARGET_WIN_RATE),
    "--eval-games", str(EVAL_GAMES),
    "--train-envs", str(TRAIN_ENVS),
    "--message", SUBMISSION_MESSAGE,
]
if SUBMIT_TO_KAGGLE:
    command.append("--submit")
if SUBMIT_BELOW_TARGET:
    command.append("--submit-below-target")
if FORCE_SUBMIT:
    command.append("--force-submit")
print("Запускаю RL pipeline...", flush=True)
subprocess.run(command, cwd=PROJECT, check=True)

In [ ]:
# Итог: метрики, файлы и состояние отправок.
import json

RESULTS = DRIVE_ROOT / "rl_boatlee_v16"
report = json.loads((RESULTS / "training_report.json").read_text())
print(json.dumps({"best": report["best"], "target_met": report["target_met"]}, indent=2))
print("\nФайлы на Drive:")
for path in sorted(RESULTS.iterdir()):
    print(f"- {path.name}: {path.stat().st_size / 1024 / 1024:.2f} MB")
if SUBMIT_TO_KAGGLE and (report["target_met"] or SUBMIT_BELOW_TARGET):
    subprocess.run(["kaggle", "competitions", "submissions", "kaggriculture"], check=True)